In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Column
from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta, timezone
import pytz
from delta.tables import *
from typing import Dict, List, Optional
from pyspark.sql.utils import AnalysisException

from zoneinfo import ZoneInfo
import re
import uuid
import copy

import json

from string import Template

import traceback
from pyspark.sql.utils import AnalysisException

import phonenumbers
from phonenumbers import PhoneNumberFormat
import pandas as pd

from graphframes import GraphFrame

In [0]:
%run ../env

In [0]:
%run ./data_constant

In [0]:
def get_ex_param(key_name, default_value):
    value = ""
    try:
        value = dbutils.widgets.get(key_name)
    except Exception as e:
        value = default_value
    
    return value

In [0]:
# formatter
# "%Y%m%d"
# "%Y-%m-%d %H:%M:%S"

def get_now_offset_cst(formatter, offset_days):
    tz = pytz.timezone("Asia/Shanghai")
    dt = (datetime.utcnow() + timedelta(days=offset_days))
    return tz.fromutc(dt).strftime(formatter)


def get_now_cst(formatter):
    return get_now_offset_cst(formatter, 0)



In [0]:
# 定义 UTC+8 时区
TZ_UTC8 = timezone(timedelta(hours=8))

def print_log(message, level="INFO"):
    """打印带 UTC+8 时间戳的日志"""
    timestamp = datetime.now(TZ_UTC8).strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")



In [0]:
def convert_colum_type(originalDf, targetDf):
    result = originalDf
    originalSchema = originalDf.schema
    targetSchema = targetDf.schema

    # Step 1: 检查 original 中的列是否在 target 中
    for originalCol in originalSchema:
        originalType = originalCol.dataType
        originalName = originalCol.name
        tar_cols = [tar_col for tar_col in targetSchema if tar_col.name.upper() == originalName.upper()]

        if len(tar_cols) == 0:
            print(f"col:{originalName} is not in tarDF")
        else:
            tar_type = tar_cols[0].dataType
            if originalType != tar_type:
                print(f"origName={originalName} ; origType={originalType} ; tarType={tar_type}")
                result = result.withColumn(originalName, F.col(originalName).cast(tar_type))

    # Step 2: 【新增】检查 target 中的列是否在 original 中
    for targetCol in targetSchema:
        targetName = targetCol.name
        orig_cols = [orig_col for orig_col in originalSchema if orig_col.name.upper() == targetName.upper()]
        if len(orig_cols) == 0:
            print(f"col:{targetName} is not in orgDF")

    return result

In [0]:
def exclude_reject_records(
    source_df,
    source_key_col
):
    # 确保 reject_df 只包含 key 列，提升性能
    reject_table = f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer_rejects"

    reject_keys = (
        spark.table(reject_table)
        .select("SRCC_ID")
        .distinct()
    )

    # 执行 left_anti join
    result_df = source_df.join(
        reject_keys,
        source_df[source_key_col] == reject_keys["SRCC_ID"],
        how="left_anti"
    )
    return result_df

In [0]:
def append_table(df, table):
    tmpDf = df

    tarDf = spark.table(f"{table}").where(F.lit(False))
    convert_colum_type(tmpDf, tarDf).write.format('delta').mode('append').option("mergeSchema", "true").saveAsTable(table)


def save_to_target_table(reqDf, tabName, whereStr="1=1"):
    
    reqDf.createOrReplaceTempView("tmpView")
    tmpDf = spark.sql(f"select * from tmpView where {whereStr}")
    tarDf = spark.table(f"{tabName}").where(F.lit(False))
    print(whereStr)
    (convert_colum_type(tmpDf, tarDf)
        .write
        .format("delta")
        .mode("overwrite")
        #.option("mergeSchema", "true")
        .option("replaceWhere", whereStr)
        .saveAsTable(f"{tabName}")
    )

#Ingest

In [0]:
def ingest_kafka_to_bronze(
    spark,
    param_source_kafka_brokers: str,
    param_source_kafka_topics: str,
    param_bronze_path: str,
    task_id:str,
    batch_id: str,
    max_topic_batch_count: int = 0,
    enable_batch_split: bool = False
):
    """
    从 Kafka 拉取自上次处理以来的新数据，写入 Delta Bronze 表（去重 + 增量合并）。

    Args:
        spark: SparkSession 实例
        param_source_kafka_brokers: Kafka broker 地址
        param_source_kafka_topics: 要订阅的 topic 列表
        param_bronze_path: Bronze 表路径
        task_id: 外部传入任务 id
        batch_id: 外部传入批次 id
        max_topic_batch_count: 单个 topic 最大分片数（用于分批）
        enable_batch_split: 是否启用分批
    """
    log_table_batch = f"{get_env_config('config_database')}.t_topic_batch_log"
    last_read_table = f"{get_env_config('config_database')}.t_kafka_last_read" 

    # ----------------------------
    # 1. 获取当前时间（UTC）
    # ----------------------------
    exec_time = datetime.utcnow()
    exec_time_ms = int(exec_time.timestamp() * 1000)

    print(f"[ingest_kafka] Brokers: {param_source_kafka_brokers}")
    print(f"[ingest_kafka] Topics: {param_source_kafka_topics}")
    print(f"[ingest_kafka] Bronze path: {param_bronze_path}")

    topics = [topic.strip() for topic in param_source_kafka_topics.split(",")]
    if len(topics) == 0:
        print("[ingest_kafka] No topics.")
        return

    # ----------------------------
    # 2. 读取每个 topic 的最新处理时间戳
    # ----------------------------
    last_read_df = spark.table(last_read_table)
    # 按 topic 取最新的 last_end_ms（基于 creation_time
    
    window_spec = Window.partitionBy("topic").orderBy(F.col("creation_time").desc())
    latest_last_read = (
        last_read_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .select("topic", "last_end_ms")
    )

    # 构建 last_end_ms 映射
    last_end_map = {}
    for row in latest_last_read.collect():
        last_end_map[row["topic"]] = row["last_end_ms"]

    # ----------------------------
    # 3. 拉取 Kafka 数据
    # ----------------------------
    raw = None
    for topic in topics:
        start_ms = last_end_map.get(topic, 0)  # 若无记录，则从 0 开始

        print(f"[ingest_kafka] Topic={topic} startingTimestamp={start_ms} endingTimestamp={exec_time_ms}")

        topic_raw = (
            spark.read.format("kafka")
            .option("kafka.bootstrap.servers", param_source_kafka_brokers)
            .option("subscribe", topic)
            .option("startingTimestamp", start_ms)
            .option("endingTimestamp", exec_time_ms)
            .option("startingOffsetsByTimestampStrategy", "latest")
            .option("failOnDataLoss", "false")
            .load()
        )

        if raw is None:
            raw = topic_raw
        else:
            raw = raw.union(topic_raw)


    # ----------------------------
    # 4. 解析并构建 Bronze 输出
    # ----------------------------
    bronze_out = raw.select(
        F.expr("uuid()").alias("slndc_id"),
        F.lit(task_id).alias("task_id"),
        F.lit(batch_id).alias("slndc_batch_id"),
        F.lit(exec_time).cast("timestamp").alias("slndc_batch_dt"),
        F.col("topic").alias("slndc_kafka_topic"),
        F.col("partition").alias("slndc_kafka_partition"),
        F.col("offset").alias("slndc_kafka_offset"),
        F.col("timestamp").alias("slndc_kafka_timestamp"),
        F.col("key").cast("string").alias("slndc_kafka_key"),
        F.col("value").cast("string").alias("slndc_payload"),
        F.current_timestamp().alias("slndc_creation_dt"),
        F.lit("").alias("slndc_creationuid"),
        F.current_timestamp().alias("slndc_update_dt"),
        F.lit("").alias("slndc_updateuid")
    )

    
    # ----------------------------
    # 5. 添加 batch_number（可选分批）
    # ----------------------------
    if enable_batch_split and max_topic_batch_count > 0:
        window_spec = Window.partitionBy("slndc_kafka_topic").orderBy(F.col("slndc_kafka_timestamp").asc())
        bronze_with_row = (
            bronze_out
            .withColumn("row_num", F.row_number().over(window_spec))
            .withColumn("batch_seq", F.ceil(F.col("row_num") / max_topic_batch_count).cast("int"))
            .withColumn("batch_seq_str", F.format_string("%04d", F.col("batch_seq")))
            .withColumn(
                "batch_number",
                F.concat_ws("_", F.col("slndc_kafka_topic"), F.lit(batch_id), F.col("batch_seq_str"))
            )
            .drop("row_num", "batch_seq", "batch_seq_str")
        )
    else:
        bronze_with_row = bronze_out.withColumn("batch_number", F.concat_ws("_", F.col("slndc_kafka_topic"), F.lit(batch_id), F.lit("0001")))

    # ----------------------------
    # 6. 写入 Delta 表
    # ----------------------------
    bronze_with_row.cache()

    (bronze_with_row.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .option("path", param_bronze_path)
    .save())

    print(f"[ingest_kafka] Appended {bronze_with_row.count()} records to bronze table at {param_bronze_path}")

    # ----------------------------
    # 7. 更新 last_read 表
    # ----------------------------
    update_last_read_table(topics, exec_time_ms, exec_time, last_read_table, batch_id, task_id, bronze_with_row)

    # ----------------------------
    # 8. 写入 batch 日志
    # ----------------------------
    if enable_batch_split and max_topic_batch_count > 0:
        write_batch_log(bronze_with_row, log_table_batch)

    bronze_with_row.unpersist()



# ========================
# 辅助函数 ：Kafka拉取记录表
# ========================
def update_last_read_table(topics, exec_time_ms, exec_time, last_read_table, batch_id, task_id, bronze_df): 
    if not topics:
        return
        
    # 构建新数据：增加 creation_time 字段（用于排序）
    creation_time = datetime.now(timezone.utc)
    
    log_data = [(task_id, batch_id, t, exec_time_ms, exec_time, creation_time) for t in topics]
    log_df = (spark
        .createDataFrame(log_data, schema="task_id STRING, batch_id STRING, topic STRING, last_end_ms BIGINT, last_end_dt TIMESTAMP, creation_time TIMESTAMP").alias("log_tab")
        .join(bronze_df.groupBy("slndc_kafka_topic").count().alias("count_tab"), F.col("log_tab.topic") == F.col("count_tab.slndc_kafka_topic"), "left")
        .select(
            F.col("log_tab.*"),
            F.coalesce(F.col("count_tab.count"), F.lit(0)).alias("read_record_count")
        ))

    # 直接 append，保留历史
    log_df.write.mode("append").saveAsTable(last_read_table)
    
    print(f"[ingest_kafka] Appended last_read history for {len(topics)} topics.")

# ========================
# 辅助函数 ：写入 batch 日志
# ========================
def write_batch_log(bronze_df, log_table):
    if bronze_df.isEmpty():
        print("[write_batch_log] Skip: bronze_df is empty.")
        return

    batch_log = (
        bronze_df
        .groupBy( "task_id", "slndc_batch_id", "slndc_kafka_topic", "batch_number")
        .agg(F.count("*").alias("read_record_count"))
        .select(
            F.col("task_id"),
            F.col("slndc_batch_id").alias("batch_id"),
            F.col("slndc_kafka_topic").alias("topic"),
            F.col("batch_number").alias("batch_number"),
            F.col("read_record_count"),
            F.current_timestamp().alias("creation_time"),
            F.lit(False).alias("is_handle")
        )
    )
    batch_log.write.mode("append").saveAsTable(log_table)
    print(f"[batch_log] Inserted {batch_log.count()} batch records into {log_table}")

In [0]:
def ingest_kafka_to_bronze_by_offset(
    spark,
    param_source_kafka_brokers: str,
    param_source_kafka_topics: str,
    param_bronze_path: str,
    checkpoint_path: str,
    kafka_groupId: str

):
    """
    从 Kafka 增量消费数据，写入 Delta Bronze 表（自动去重 + 增量合并）。
    
    使用 Structured Streaming + foreachBatch，通过 checkpoint 自动管理 offset。
    
    Args:
        spark: SparkSession 实例
        param_source_kafka_brokers: Kafka broker 地址，如 "host1:9092,host2:9092"
        param_source_kafka_topics: 要订阅的 topic 列表，如 "topic1,topic2"
        param_bronze_path: Bronze Delta 表的存储路径（dbfs/s3/abfss 等）
        checkpoint_path: Checkpoint 路径（必须提供，用于 offset 和状态管理）
        kafka_groupId: 消费者组 ID
    """
    # 生成唯一 queryName
    unique_suffix = uuid.uuid4().hex[:8]  # 例如: 'a1b2c3d4'
    query_name = f"ingest_kafka_to_bronze_{unique_suffix}"
    print(f"[ingest_kafka] Query name: {query_name}")

    # 判断 bronze 表是否存在
    try:
        dbutils.fs.ls(param_bronze_path)
        bronze_exists = True
    except Exception:
        bronze_exists = False

    print(f"[ingest_kafka] Brokers: {param_source_kafka_brokers}")
    print(f"[ingest_kafka] Topics: {param_source_kafka_topics}")
    print(f"[ingest_kafka] Bronze path: {param_bronze_path}")
    print(f"[ingest_kafka] Checkpoint path: {checkpoint_path}")
    print(f"[ingest_kafka] Group Id: {kafka_groupId}")

    def micro_batch_output(micro_batch_df, batch_id):
        if micro_batch_df.isEmpty():
            print(f"[Batch {batch_id}] No new records from Kafka.")
            return

        # 解析 Kafka 消息并构建 Bronze 输出结构
        bronze_out = micro_batch_df.select(
            F.expr("uuid()").alias("slndc_id"),
            F.date_format(F.current_timestamp(), "yyyyMMddHHmmss").alias("slndc_batch_id"),
            F.current_timestamp().alias("slndc_batch_dt"),
            F.lit("ConsumerList").alias("slndc_object"),
            F.col("topic").alias("slndc_kafka_topic"),
            F.col("partition").alias("slndc_kafka_partition"),
            F.col("offset").alias("slndc_kafka_offset"),
            F.col("timestamp").alias("slndc_kafka_timestamp"),  # 单位：毫秒
            F.col("value").cast("string").alias("slndc_payload"),
            F.current_timestamp().alias("slndc_creation_dt"),
            F.lit("").alias("slndc_creationuid"),
            F.current_timestamp().alias("slndc_update_dt"),
            F.lit("").alias("slndc_updateuid")
        )

        # 写入 Delta 表：首次创建 or 后续 upsert
        if not bronze_exists:
            (bronze_out.write
             .format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .save(param_bronze_path))
            print(f"[Batch {batch_id}] Created and appended to new bronze table.")
        else:
            dt = DeltaTable.forPath(spark, param_bronze_path)
            merge_condition = """
                t.slndc_kafka_topic = s.slndc_kafka_topic AND
                t.slndc_kafka_partition = s.slndc_kafka_partition AND
                t.slndc_kafka_offset = s.slndc_kafka_offset AND
                t.slndc_kafka_timestamp = s.slndc_kafka_timestamp
            """
            result = (
                dt.alias("t")
                .merge(bronze_out.alias("s"), merge_condition)
                .whenNotMatchedInsertAll()
                .execute()
            )
            inserted = result.numTargetRowsInserted()
            print(f"[Batch {batch_id}] Merged {inserted} new records into existing bronze table.")

    # 启动一次性流式作业（处理所有当前可用数据）
    kafka_stream = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", param_source_kafka_brokers)
        .option("subscribe", param_source_kafka_topics)
        .option("kafka.group.id", kafka_groupId)
        .option("startingOffsets", "earliest")  # 首次运行时生效，后续由 checkpoint 控制
        .option("failOnDataLoss", "false")
        .load()
    )

    query = (
        kafka_stream
        .writeStream
        .queryName(query_name)
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .foreachBatch(micro_batch_output)
        .trigger(availableNow=True)  # 一次性触发，适合批调度
        .start()
    )

    query.awaitTermination()
    print("[ingest_kafka] Ingestion completed.")

In [0]:
def load_csv_data_from_blob(blob_storage_name, blob_storage_key, csv_file_path):
    spark.conf.set(f"fs.azure.account.key.{blob_storage_name}.blob.core.windows.net", blob_storage_key)
    return spark.read.option("header", "true").option("inferSchema", "true").csv(csv_file_path)

#Merge

In [0]:
# ['a','b']
# [('a','a1'),('b','b2')]
def get_merge_condition(merge_cols, is_same_field=True):
    if merge_cols is None or len(merge_cols)==0:
        return None
    if is_same_field:
        return " AND ".join([f"b.{col} = u.{col}" for col in merge_cols])
    else:
        return " AND ".join([f"b.{col_map[0]} = u.{col_map[1]}" for col_map in merge_cols])



# ['a','b']
# [('a','a1'),('b','b2')]
def get_update_condition(update_cols, is_same_field=True):
    if update_cols is None or len(update_cols)==0:
        return None
    if is_same_field:
        return " OR ".join([f"coalesce( string(b.{col}), '_DEFAULT_VALUE') != coalesce( string(u.{col}) , '_DEFAULT_VALUE')" for col in update_cols])
    else:
        return " OR ".join([f"coalesce( string(b.{col_map[0]}), '_DEFAULT_VALUE') != coalesce( string(u.{col_map[1]}), '_DEFAULT_VALUE')" for col_map in update_cols])



# Merge T table
def merge_t_table(t_table_name, update_df, merge_condition, update_condition=None):
    # base table
    base_table = DeltaTable.forName(spark, t_table_name)
    if base_table:
        # merge
        print("Merging with full load...")
        (base_table.alias("b")
            .merge(update_df.alias("u"), merge_condition)
            .whenMatchedUpdateAll(condition = update_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )


In [0]:
# get latest version
def get_latest_version(t_table_name):
    vdf = spark.sql(f'''
                SELECT version
                FROM (DESCRIBE HISTORY {t_table_name}) 
                WHERE operation = 'MERGE' OR operation = 'WRITE' OR operation = 'CREATE OR REPLACE TABLE AS SELECT' 
                ORDER BY version DESC
                '''
        )
    return vdf.select("version").collect()[0][0]

In [0]:
def append_step_log(
    log_table_name: str,
    task_id: str,
    step_num: str,
    step_name: str,
    start_time,
    end_time,
    status: str,
    message: str,
    project: str 
):
    """
    写入步骤日志（无幂等，append）
    字段:
    id, project, task_id, step_num, step_name, start_time, end_time, status, message
    """
    status = (status or "").upper()
    if status not in ("SUCCESS", "FAILED", "WARNING"):
        raise ValueError(f"Invalid status: {status}, only SUCCESS/FAILED/WARNING allowed")

    schema = StructType([
        StructField("id", StringType(), False),
        StructField("project", StringType(), False),
        StructField("task_id", StringType(), True),
        StructField("step_num", StringType(), False),
        StructField("step_name", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("message", StringType(), True),
    ])

    row = [(
        str(uuid.uuid4()),
        project,
        str(task_id),
        str(step_num),
        step_name,
        start_time,
        end_time,
        status,
        (message or "")[:2000]
    )]

    log_df = spark.createDataFrame(row, schema)
    append_table(log_df, log_table_name)

In [0]:
class StepLogger:
    """Context manager wrapping try/except/finally + append_step_log pattern.

    Usage:
        with StepLogger("step_name", "step_num", "project", task_id=task_id) as logger:
            main_function(task_id)

        # For touchpoint projects using batch_id as task_id:
        with StepLogger("step_name", "step_num", "touchpoint", task_id=batch_id) as logger:
            main()

        # WARNING status for non-fatal issues:
        with StepLogger("step_name", "step_num", "project", task_id=task_id) as logger:
            main_function(task_id)
            if some_non_critical_condition:
                logger.status = "WARNING"
                logger.message = "completed with warnings: ..."
    """
    def __init__(self, step_name, step_num, project, task_id=None):
        self.step_name = step_name
        self.step_num = step_num
        self.project = project
        self.task_id = task_id
        self.log_table_name = f"{get_env_config('config_database')}.t_task_step_log"
        self.start_time = None
        self.status = "SUCCESS"
        self.message = "completed"

    def __enter__(self):
        self.start_time = datetime.now()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        end_time = datetime.now()
        if exc_type is not None:
            self.status = "FAILED"
            self.message = f"{exc_type.__name__}: {str(exc_val)}"
        append_step_log(
            log_table_name=self.log_table_name,
            task_id=self.task_id,
            step_num=self.step_num,
            step_name=self.step_name,
            start_time=self.start_time,
            end_time=end_time,
            status=self.status,
            message=self.message,
            project=self.project,
        )
        return False  # always propagate exceptions (re-raise on failure)

#ukey group

In [0]:
def get_ukeyGroup_with_flag(task_id):
    ukey_df = (spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group")
            .filter(F.col("task_id") == task_id)
    )

    win_master = Window.partitionBy("mrkt_code", "master_consumermdmkey")
    win_cid = Window.partitionBy("mrkt_code", "brnd_code", "source_code", "consumer_id")

    flag_df = ukey_df.withColumn(
        "is_masterUkey_duplicate",
        F.when(
            (F.col("match_type") == MATCH_TYPE_REGULAR_STR) & (F.col("is_master_recode") == True),
            F.size(F.collect_set("new_consumermdmkey").over(win_master)) > 1,
        ).otherwise(F.lit(False)),
    ).withColumn(
        "is_cid_duplicate",
        F.when(
            F.col("match_type") == MATCH_TYPE_REGULAR_STR,
            F.size(F.collect_set("new_consumermdmkey").over(win_cid)) > 1,
        ).otherwise(F.lit(False)),
    )

    return flag_df

In [0]:
def get_ukey_group_by_process(task_id):
    process_df = (get_ukeyGroup_with_flag(task_id)
        .filter(F.coalesce(F.col("is_masterUkey_duplicate"), F.lit(False)) != True)
        .filter(F.coalesce(F.col("is_cid_duplicate"), F.lit(False)) != True)
        .drop("is_masterUkey_duplicate", "is_cid_duplicate")
    )

    return process_df

#Cid group

In [0]:
def get_cidGroup_with_flag(task_id):
    ukey_df = (spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_cid_group")
            .filter(F.col("task_id") == task_id)
    )

    win_master = Window.partitionBy("mrkt_code", "mapping_conusmer_id")
    win_order = Window.partitionBy("mrkt_code", "brnd_code", "source_code", "order_id")

    flag_df = ukey_df.withColumn(
        "is_masterCid_duplicate",
        F.when(
            (F.col("match_type") == MATCH_TYPE_REGULAR_STR) & (F.col("record_type") == CID_MATCH_RECORD_TYPE_CM),
            F.size(F.collect_set("new_mapping_conusmer_id").over(win_master)) > 1,
        ).otherwise(F.lit(False)),
    ).withColumn(
        "is_order_duplicate",
        F.when(
            (F.col("match_type") == MATCH_TYPE_REGULAR_STR) & (F.col("order_id").isNotNull()),
            F.size(F.collect_set("new_mapping_conusmer_id").over(win_order)) > 1,
        ).otherwise(F.lit(False)),
    )

    return flag_df

In [0]:
def get_cidGroup_by_process(task_id):
    process_df = (get_cidGroup_with_flag(task_id)
        .filter(F.coalesce(F.col("is_masterCid_duplicate"), F.lit(False)) != True)
        .filter(F.coalesce(F.col("is_order_duplicate"), F.lit(False)) != True)
        .drop("is_masterCid_duplicate", "is_order_duplicate")
    )

    return process_df